# 💬 Cookbook 3: Conversations & Multi-Agent Systems

In this cookbook, you'll learn how to manage conversations and build multi-agent systems!

**Topics covered:**
1. Multi-turn conversations and context
2. Conversation management strategies
3. Agent state management
4. Multi-agent patterns (agents as tools, swarm, graph)
5. Building coordinated agent systems

**Prerequisites**: Complete Cookbooks 1-2 first!

---

In [ ]:
# Install packages
!pip install strands-agents strands-agents-tools -q
print("✅ Ready to go!")

---
## Part 1: Multi-Turn Conversations
---

### 💬 Basic Conversation Context

Agents automatically maintain conversation history:

In [ ]:
from strands import Agent

agent = Agent(
    system_prompt="You are a helpful travel assistant."
)

# First message
agent("I'm planning a trip to Japan")

# Follow-up - agent remembers!
agent("What's the best time to visit?")

# Another follow-up
agent("What about food recommendations?")

print(f"\n📊 Total messages: {len(agent.messages)}")

### 🔍 Viewing Conversation History

In [ ]:
from strands import Agent

agent = Agent(system_prompt="You are a concise assistant.")

agent("My name is Alice")
agent("What's my name?")

# View all messages
print("=== Conversation History ===")
for i, msg in enumerate(agent.messages):
    role = msg.get("role", "unknown")
    content = msg.get("content", [])
    
    # Extract text from content blocks
    text = ""
    for block in content:
        if isinstance(block, dict) and "text" in block:
            text = block["text"][:100] + "..." if len(block.get("text", "")) > 100 else block.get("text", "")
            break
    
    print(f"{i}. [{role}]: {text}")

### 🔄 Resetting Conversations

In [ ]:
from strands import Agent

agent = Agent(system_prompt="You are a helpful assistant.")

# Have a conversation
agent("My favorite color is blue")
agent("What's my favorite color?")
print(f"Messages before reset: {len(agent.messages)}")

# Reset the conversation
agent.messages.clear()
print(f"Messages after reset: {len(agent.messages)}")

# Agent no longer remembers
agent("What's my favorite color?")
print("\n✅ Conversation reset - agent doesn't remember previous context!")

---
## Part 2: Conversation Management Strategies
---

As conversations grow long, you need strategies to manage context window limits.

### 📏 Sliding Window Strategy

Keep only the most recent N messages:

In [ ]:
from strands import Agent
from strands.agent.conversation_manager.sliding_window_conversation_manager import SlidingWindowConversationManager

# Keep only the last 10 messages
conversation_manager = SlidingWindowConversationManager(window_size=10)

agent = Agent(
    system_prompt="You are a helpful assistant.",
    conversation_manager=conversation_manager
)

# Simulate a long conversation
for i in range(15):
    agent(f"This is message {i+1}")

print(f"\n📊 Total messages (with sliding window): {len(agent.messages)}")
print("Oldest messages were automatically removed!")

### 📝 Summarizing Strategy

Summarize old messages to preserve context while reducing tokens:

In [ ]:
from strands import Agent
from strands.agent.conversation_manager.summarizing_conversation_manager import SummarizingConversationManager

# Summarize when conversation gets long
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.5  # Keep 50% of messages, summarize rest
)

agent = Agent(
    system_prompt="You are a helpful assistant.",
    conversation_manager=conversation_manager
)

# Have a longer conversation
agent("I'm building an e-commerce app")
agent("It needs user authentication")
agent("And a shopping cart feature")
agent("With payment integration")
agent("Can you summarize what we've discussed?")

print("\n✅ Summarizing conversation manager active!")

---
## Part 3: Agent State Management
---

Agents can store custom key-value state for tracking data across invocations.

In [ ]:
from strands import Agent, tool, ToolContext

@tool(context=True)
def remember_preference(key: str, value: str, tool_context: ToolContext) -> str:
    """Store a user preference in agent state.
    
    Args:
        key: The preference name.
        value: The preference value.
        tool_context: Context for state access.
    
    Returns:
        Confirmation message.
    """
    # Store in agent state
    tool_context.agent.state[key] = value
    return f"Stored preference: {key} = {value}"

@tool(context=True)
def get_preference(key: str, tool_context: ToolContext) -> str:
    """Retrieve a stored preference.
    
    Args:
        key: The preference name to retrieve.
        tool_context: Context for state access.
    
    Returns:
        The stored value or 'not found'.
    """
    value = tool_context.agent.state.get(key, "not found")
    return f"Preference {key}: {value}"

@tool(context=True)
def list_preferences(tool_context: ToolContext) -> dict:
    """List all stored preferences.
    
    Args:
        tool_context: Context for state access.
    
    Returns:
        All stored preferences.
    """
    return dict(tool_context.agent.state)

# Create agent with state-aware tools
agent = Agent(
    tools=[remember_preference, get_preference, list_preferences],
    system_prompt="You help users manage their preferences. Use the tools to store and retrieve them."
)

# Test it
agent("Remember that my favorite color is blue and my preferred language is Python")
agent("What are all my preferences?")

# Check state directly
print(f"\nDirect state access: {dict(agent.state)}")

---
## Part 4: Multi-Agent Systems ⭐
---

Multiple agents can work together to solve complex problems!

### 🔧 Agents as Tools

The simplest multi-agent pattern - use one agent as a tool for another:

In [ ]:
from strands import Agent

# Create specialist agents
math_agent = Agent(
    name="MathExpert",
    system_prompt="You are a math expert. Solve math problems step by step."
)

writing_agent = Agent(
    name="Writer",
    system_prompt="You are a creative writer. Write engaging content."
)

# Orchestrator agent uses specialists as tools
orchestrator = Agent(
    name="Orchestrator",
    system_prompt="""You coordinate between specialists.
    Use MathExpert for math problems and Writer for creative content.""",
    tools=[math_agent.as_tool(), writing_agent.as_tool()]
)

# Test it
response = orchestrator(
    "Calculate 15% of 280, then write a haiku about that number"
)

print("\n✅ Orchestrator coordinated between specialists!")

### 🐝 Swarm Pattern

A swarm is a group of agents that can hand off tasks to each other:

In [ ]:
from strands import Agent
from strands.multiagent import Swarm

# Create specialized agents
greeter = Agent(
    name="greeter",
    system_prompt="""You are a friendly greeter. Welcome users and understand their needs.
    If they need technical help, hand off to 'tech_support'.
    If they need billing help, hand off to 'billing'."""
)

tech_support = Agent(
    name="tech_support",
    system_prompt="""You are a technical support expert. Help with technical issues.
    If the issue is billing-related, hand off to 'billing'."""
)

billing = Agent(
    name="billing",
    system_prompt="""You are a billing specialist. Help with payment and subscription issues."""
)

# Create a swarm
support_swarm = Swarm(
    agents=[greeter, tech_support, billing],
    default_agent=greeter  # Start with greeter
)

# Run the swarm
result = support_swarm(
    "Hi, I'm having trouble with my subscription payment"
)

print(f"\n✅ Final response from: {result}")

### 📊 Graph Pattern

Define explicit workflows between agents:

In [ ]:
from strands import Agent
from strands.multiagent import Graph, GraphBuilder

# Create agents for a content pipeline
researcher = Agent(
    name="researcher",
    system_prompt="You research topics and gather key facts. Be thorough but concise."
)

writer = Agent(
    name="writer",
    system_prompt="You write engaging content based on research provided."
)

editor = Agent(
    name="editor",
    system_prompt="You review and improve written content. Fix grammar and improve clarity."
)

# Build the graph
builder = GraphBuilder()

# Add nodes
builder.add_node("research", researcher)
builder.add_node("write", writer)
builder.add_node("edit", editor)

# Define edges (workflow)
builder.add_edge("research", "write")  # research → write
builder.add_edge("write", "edit")       # write → edit

# Set entry and exit points
builder.set_entry_point("research")
builder.set_exit_point("edit")

# Build the graph
content_pipeline = builder.build()

# Run it!
result = content_pipeline("Write a short blog post about the benefits of meditation")

print(f"\n✅ Content pipeline complete!")

### 🔄 Cyclic Graph (with Router)

Create loops in your workflow with conditional routing:

In [ ]:
from strands import Agent
from strands.multiagent import Graph, GraphBuilder
from typing import Literal

# Agents for a review workflow
draft_writer = Agent(
    name="draft_writer",
    system_prompt="Write a draft response. Keep it brief."
)

quality_checker = Agent(
    name="quality_checker",
    system_prompt="""Review the draft and respond with exactly:
    - 'APPROVED' if the draft is good
    - 'NEEDS_REVISION' if it needs improvement"""
)

# Router function to decide next step
def quality_router(state) -> Literal["revise", "done"]:
    """Route based on quality check result."""
    # Check the last message for approval status
    last_response = str(state.get("last_response", ""))
    if "APPROVED" in last_response.upper():
        return "done"
    return "revise"

# Build graph with cycle
builder = GraphBuilder()

builder.add_node("write", draft_writer)
builder.add_node("check", quality_checker)

# Add conditional routing
builder.add_edge("write", "check")
builder.add_conditional_edges("check", quality_router, {
    "revise": "write",  # Loop back to write
    "done": None         # Exit the graph
})

builder.set_entry_point("write")

review_graph = builder.build()

# Run it (may loop if quality check fails)
result = review_graph("Explain quantum computing in one sentence")

print("\n✅ Review workflow complete!")

---
## Part 5: Multi-Agent Debate Pattern
---

Multiple agents can debate to reach better conclusions:

In [ ]:
from strands import Agent

# Create debate agents with different perspectives
optimist = Agent(
    name="optimist",
    system_prompt="""You are an optimist who sees the positive side of things.
    Present arguments for why ideas will succeed. Be enthusiastic but substantive."""
)

skeptic = Agent(
    name="skeptic", 
    system_prompt="""You are a skeptic who identifies potential problems.
    Present arguments for why ideas might fail. Be critical but fair."""
)

synthesizer = Agent(
    name="synthesizer",
    system_prompt="""You synthesize different viewpoints into a balanced conclusion.
    Consider both optimistic and skeptical perspectives."""
)

# Simple debate function
def debate(topic: str, rounds: int = 2) -> str:
    """Run a multi-round debate on a topic."""
    
    print(f"🎯 Topic: {topic}\n")
    
    arguments = []
    
    for round_num in range(1, rounds + 1):
        print(f"=== Round {round_num} ===")
        
        # Optimist argues
        opt_prompt = f"Topic: {topic}\nPrevious arguments: {arguments}\nMake your optimistic case (2-3 sentences):"
        opt_response = optimist(opt_prompt)
        arguments.append(f"Optimist: {opt_response}")
        print(f"😊 Optimist: {opt_response}\n")
        
        # Skeptic argues
        skep_prompt = f"Topic: {topic}\nPrevious arguments: {arguments}\nMake your skeptical case (2-3 sentences):"
        skep_response = skeptic(skep_prompt)
        arguments.append(f"Skeptic: {skep_response}")
        print(f"🤔 Skeptic: {skep_response}\n")
    
    # Synthesize
    print("=== Synthesis ===")
    synth_prompt = f"Topic: {topic}\nArguments from debate:\n{chr(10).join(arguments)}\n\nProvide a balanced conclusion:"
    synthesis = synthesizer(synth_prompt)
    print(f"⚖️ Synthesis: {synthesis}")
    
    return str(synthesis)

# Run a debate
result = debate("Should AI be used to write code in production?")

---
## 🏆 Challenge: Build a Research Pipeline

Create a multi-agent system that:
1. Takes a research question
2. Has a researcher gather information
3. Has an analyst identify key insights
4. Has a writer create a summary

In [ ]:
# Your code here!
from strands import Agent
from strands.multiagent import GraphBuilder

# Create your agents
researcher = Agent(
    name="researcher",
    system_prompt="Your system prompt..."
)

analyst = Agent(
    name="analyst",
    system_prompt="Your system prompt..."
)

writer = Agent(
    name="writer",
    system_prompt="Your system prompt..."
)

# Build your graph
builder = GraphBuilder()
# Add your nodes and edges...

# Test it
# pipeline = builder.build()
# result = pipeline("Your research question...")

---
## 📚 Summary

In this cookbook, you learned:

1. **Conversations** - Multi-turn context, viewing history, resetting
2. **Conversation Management** - Sliding window and summarizing strategies
3. **Agent State** - Custom key-value storage across invocations
4. **Agents as Tools** - Using one agent as a tool for another
5. **Swarm Pattern** - Agents that hand off tasks to each other
6. **Graph Pattern** - Explicit workflows with conditional routing
7. **Debate Pattern** - Multiple perspectives synthesized together

**Next**: Check out Cookbook 4 for production patterns - memory, hooks, and deployment!